# Практическая работа № 3. Сборка пайплайна предобработки

ФИО: Эркинбеков Айман  
Вариант: B

Продолжаю работу с Titanic из второй практической.

Цель: собрать обработку числовых и категориальных признаков в один пайплайн. Все параметры обработки нужно получить только по обучающей части.

Целевая переменная: Survived. Значение 0 означает, что пассажир не выжил, 1 означает, что выжил.

В этой работе модель пока не обучаю, только готовлю данные.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler

print("Библиотеки загружены")

Библиотеки загружены


In [2]:
DATA_PATH = Path("../data/train.csv")

assert DATA_PATH.exists(), f"Файл не найден: {DATA_PATH}"

df = pd.read_csv(DATA_PATH)

print("Размер таблицы:", df.shape)
print("Столбцы:", df.columns.tolist())

Размер таблицы: (891, 12)
Столбцы: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


In [3]:
X = df.drop(columns=["Survived"])
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (712, 11)
X_test: (179, 11)
y_train: (712,)
y_test: (179,)


Данные разделены до заполнения пропусков и масштабирования. В обучающей части 712 строк, в проверочной 179. Survived вынесен отдельно и не входит в признаки.

Использовал stratify, чтобы сохранить примерно одинаковые доли классов в обеих частях.

In [4]:
train_info = pd.DataFrame({
    "Тип в pandas": X_train.dtypes.astype(str),
    "Пропуски": X_train.isna().sum(),
    "Разных значений": X_train.nunique()
})

display(train_info)

,Тип в pandas,Пропуски,Разных значений
PassengerId,int64,0,712
Pclass,int64,0,3
Name,str,0,712
Sex,str,0,2
Age,float64,137,85
SibSp,int64,0,7
Parch,int64,0,7
Ticket,str,0,571
Fare,float64,0,226
Cabin,str,552,127


## Какие признаки использую и как обрабатываю

| Признак | Тип | Обработка и причина |
| --- | --- | --- |
| Age | Числовой, возраст | Заполняю медианой train. Она меньше зависит от крайних значений, чем среднее. Затем масштабирую. |
| Fare | Числовой, стоимость билета | При пропусках использую медиану train, так как распределение скошено и есть дорогие билеты. Затем масштабирую. |
| SibSp | Числовой дискретный, число братьев, сестёр и супругов | При пропусках использую медиану train. Это простой вариант для счётчика, который не увеличивается из-за нескольких больших семей. Затем масштабирую. |
| Parch | Числовой дискретный, число родителей и детей | При пропусках использую медиану train по той же причине, что и для SibSp. Затем масштабирую. |
| Pclass | Категориальный порядковый, класс билета | Кодирую через one-hot, чтобы не задавать числовое расстояние между классами. В исходном файле значения 1, 2 и 3. |
| Sex | Категориальный, пол | Пропуски заменяю отдельной категорией Unknown, затем применяю one-hot. |
| Embarked | Категориальный, порт посадки | Пропуски заменяю Unknown, чтобы не приписывать пассажиру порт без данных. Затем применяю one-hot. |
| PassengerId | Идентификатор | Исключаю из признаков. |
| Name | Текст, имя пассажира | В этой версии исключаю. Извлечение обращений из имени оставляю для дальнейшей работы. |
| Ticket | Категориальный, номер билета | В этой версии исключаю из-за большого числа разных значений. |
| Cabin | Категориальный, номер каюты | Пока исключаю: в train отсутствует 552 значения. Позже можно попробовать признак наличия каюты или палубу. |
| Survived | Бинарная целевая переменная | Храню отдельно в y, не передаю в предобработку. |

Для категориальных признаков использую отдельную категорию Unknown. Для Pclass это запасная стратегия на случай пропусков, в текущих данных их нет.

Во второй работе рассматривал заполнение Embarked самым частым портом. Здесь выбрал Unknown, чтобы сохранить информацию о том, что порт неизвестен.

Кандидатов в выбросы автоматически не удаляю. Большие семьи и дорогие билеты могут быть реальными наблюдениями. Для числовых признаков использую RobustScaler, который рассчитывает масштаб по медиане и межквартильному размаху. Он не удаляет выбросы.

Преобразование Fare через log1p и новые признаки из Cabin пока не добавляю. Это возможные варианты для следующих экспериментов.

In [5]:
numeric_features = ["Age", "SibSp", "Parch", "Fare"]
categorical_features = ["Pclass", "Sex", "Embarked"]

print("Числовые:", numeric_features)
print("Категориальные:", categorical_features)

Числовые: ['Age', 'SibSp', 'Parch', 'Fare']
Категориальные: ['Pclass', 'Sex', 'Embarked']


In [6]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

print("Числовой пайплайн создан")

Числовой пайплайн создан


In [7]:
categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="Unknown"
        )
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])

print("Категориальный пайплайн создан")

Категориальный пайплайн создан


In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)

print("Общий пайплайн создан")

Общий пайплайн создан


In [9]:
X_train_ready = preprocessor.fit_transform(X_train)

print("Train после обработки:", X_train_ready.shape)

Train после обработки: (712, 13)


In [10]:
fitted_numeric = preprocessor.named_transformers_["num"]

medians_check = pd.DataFrame({
    "Медиана train": X_train[numeric_features].median(),
    "Медиана в пайплайне":
        fitted_numeric.named_steps["imputer"].statistics_
})

display(medians_check)

assert np.allclose(
    medians_check["Медиана train"],
    medians_check["Медиана в пайплайне"]
)

print("Медианы совпадают с train")

,Медиана train,Медиана в пайплайне
Age,28.5000,28.5000
SibSp,0.0000,0.0000
Parch,0.0000,0.0000
Fare,14.4542,14.4542


Медианы совпадают с train


In [11]:
fitted_categorical = preprocessor.named_transformers_["cat"]

medians_before = (
    fitted_numeric.named_steps["imputer"].statistics_.copy()
)
center_before = (
    fitted_numeric.named_steps["scaler"].center_.copy()
)
scale_before = (
    fitted_numeric.named_steps["scaler"].scale_.copy()
)
categories_before = [
    values.copy()
    for values in fitted_categorical.named_steps["onehot"].categories_
]

X_test_ready = preprocessor.transform(X_test)

assert np.array_equal(
    medians_before,
    fitted_numeric.named_steps["imputer"].statistics_
)
assert np.array_equal(
    center_before,
    fitted_numeric.named_steps["scaler"].center_
)
assert np.array_equal(
    scale_before,
    fitted_numeric.named_steps["scaler"].scale_
)

for before, after in zip(
    categories_before,
    fitted_categorical.named_steps["onehot"].categories_
):
    assert np.array_equal(before, after)

print("Test после обработки:", X_test_ready.shape)
print("Параметры после обработки test не изменились")

Test после обработки: (179, 13)
Параметры после обработки test не изменились


In [12]:
feature_names = preprocessor.get_feature_names_out()

assert X_train_ready.shape[0] == len(X_train)
assert X_test_ready.shape[0] == len(X_test)

assert X_train_ready.shape[1] == X_test_ready.shape[1]
assert X_train_ready.shape[1] == len(feature_names)

assert np.isfinite(X_train_ready).all()
assert np.isfinite(X_test_ready).all()

assert set(X_train.index).isdisjoint(X_test.index)
assert "Survived" not in X_train.columns

print("Train:", X_train_ready.shape)
print("Test:", X_test_ready.shape)
print("Число выходных признаков:", len(feature_names))
print("Пропусков и бесконечных значений нет")
print("Строки train и test не пересекаются")
print("Все проверки пройдены")

Train: (712, 13)
Test: (179, 13)
Число выходных признаков: 13
Пропусков и бесконечных значений нет
Строки train и test не пересекаются
Все проверки пройдены


In [13]:
train_ready_df = pd.DataFrame(
    X_train_ready,
    columns=feature_names,
    index=X_train.index
)

display(train_ready_df.head())
print(feature_names.tolist())

,num__Age,num__SibSp,num__Parch,num__Fare,cat__Pclass_1,cat__Pclass_2,cat__Pclass_3,cat__Sex_female,cat__Sex_male,cat__Embarked_C,cat__Embarked_Q,cat__Embarked_S,cat__Embarked_Unknown
692,0.000000,0.0,0.0,1.819652,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
481,0.000000,0.0,0.0,-0.625609,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
527,0.000000,0.0,0.0,8.973477,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
855,-0.750000,0.0,1.0,-0.220921,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0
801,0.178571,1.0,1.0,0.510548,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


['num__Age', 'num__SibSp', 'num__Parch', 'num__Fare', 'cat__Pclass_1', 'cat__Pclass_2', 'cat__Pclass_3', 'cat__Sex_female', 'cat__Sex_male', 'cat__Embarked_C', 'cat__Embarked_Q', 'cat__Embarked_S', 'cat__Embarked_Unknown']


## Проверка пайплайна и выводы

1. Разделил исходные данные до заполнения пропусков и масштабирования. Получилось 712 строк в train и 179 в test.

2. Для Age, SibSp, Parch и Fare использовал заполнение медианой и RobustScaler. Медианы пайплайна совпадают с медианами обучающей части.

3. Для Pclass, Sex и Embarked использовал отдельную категорию Unknown для пропусков и one-hot кодирование. Список категорий получен только по train.

4. PassengerId, Name, Ticket и Cabin исключил из этой версии обработки. Survived хранится отдельно как целевая переменная.

5. Пайплайн обучен через fit_transform только на X_train. Для X_test использован transform. После обработки test медианы, параметры масштабирования и категории не изменились.

6. После обработки train имеет размер (712, 13), test имеет размер (179, 13). Пропусков и бесконечных значений нет. Все строки сохранены, кандидатов в выбросы автоматически не удалял.

## Ограничения

Во второй работе я уже смотрел весь датасет. Для учебного задания это продолжение EDA, но при реальной оценке модели тестовую часть лучше отделять до подробного анализа и выбора обработки.

Разбиение по пассажирам также не гарантирует, что родственники или пассажиры с общим билетом окажутся только в одной части. Для более строгой проверки модели позже стоит рассмотреть разделение по группам.